In [ ]:
import os
from pathlib import Path


def find_repo_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "downstream_tasks/expression_prediction").is_dir():
            return path
    raise RuntimeError("Run inside the GENA-LM clone or set GENA_HOME")


REPO_ROOT = (
    Path(os.environ["GENA_HOME"]).resolve()
    if "GENA_HOME" in os.environ
    else find_repo_root()
)
BENCHMARK_ROOT = Path(os.environ.get("BENCHMARK_ROOT", REPO_ROOT)).resolve()
TASK_ROOT = Path(os.environ.get("TASK_ROOT", REPO_ROOT)).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", REPO_ROOT / "data")).resolve()


In [ ]:
import json
from pathlib import Path

import pandas as pd

summary_path = Path("../data/qnorm_alphagenome_support_summary_mouse.csv")
metadata_path = Path("../data/alphagenome_mouse_rna_seq_metadata.csv")
output_path = Path("../data/alphagenome_track_to_qnorm_id_mouse_first.json")

summary = pd.read_csv(summary_path, dtype=str).fillna("")
metadata = pd.read_csv(metadata_path, dtype=str).fillna("")

summary["track_name"] = (
    summary["ontology_curie"].str.strip()
    + " "
    + summary["assay"].str.strip()
)

valid_tracks = set(metadata["name"])

supported = summary[
    (summary["alphagenome_supported"] == "yes")
    & (summary["id"] != "")
    & (summary["track_name"].isin(valid_tracks))
].copy()

first = supported.drop_duplicates("track_name", keep="first")

mapping = dict(zip(first["track_name"], first["id"]))

output_path.write_text(
    json.dumps(mapping, indent=4, sort_keys=True) + "\n"
)

print(f"Wrote: {output_path}")
print(f"Exact AlphaGenome tracks: {len(mapping)}")
print(f"Supported qnorm rows before selecting first: {len(supported)}")

In [ ]:
summary_path = Path("../data/qnorm_alphagenome_support_summary_mouse.csv")
metadata_path = Path("../data/alphagenome_mouse_rna_seq_metadata.csv")
output_path = Path("../data/alphagenome_track_to_qnorm_ids_mouse_all.json")

summary = pd.read_csv(summary_path, dtype=str).fillna("")
metadata = pd.read_csv(metadata_path, dtype=str).fillna("")

summary["track_name"] = (
    summary["ontology_curie"].str.strip()
    + " "
    + summary["assay"].str.strip()
)

valid_tracks = set(metadata["name"])

supported = summary[
    (summary["alphagenome_supported"] == "yes")
    & (summary["id"] != "")
    & (summary["track_name"].isin(valid_tracks))
].copy()

mapping = {
    track_name: sorted(set(group["id"]))
    for track_name, group in supported.groupby("track_name", sort=True)
}

output_path.write_text(
    json.dumps(mapping, indent=4, sort_keys=True) + "\n"
)

print(f"Wrote: {output_path}")
print(f"Exact AlphaGenome tracks: {len(mapping)}")
print(f"Total unique qnorm IDs: {sum(map(len, mapping.values()))}")

In [ ]:
from pathlib import Path

import pandas as pd

data_dir = DATA_ROOT

source = data_dir / "true_mouse_all_genes_qnorm_samples_by_genes.WITH_TEST.csv"
expression = pd.read_csv(source)

expression = expression.set_index("id")


def make_split(split):
    forward = pd.read_csv(
        data_dir / f"mouse.{split}.forward.csv",
        sep="\t",
        dtype=str,
    )
    reverse = pd.read_csv(
        data_dir / f"mouse.{split}.reverse.csv",
        sep="\t",
        dtype=str,
    )

    genes = (
        pd.concat([forward, reverse], ignore_index=True)["gene_id"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    available = [gene for gene in genes if gene in expression.columns]
    missing = [gene for gene in genes if gene not in expression.columns]

    # New orientation:
    # rows = genes, columns = qnorm IDs
    truth = expression[available].T
    truth.index.name = "gene_id"
    truth = truth.reset_index()

    output = data_dir / f"{split}_true_mouse.csv"
    truth.to_csv(output, index=False)

    print(f"{split}: requested={len(genes)}")
    print(f"{split}: written={len(available)}")
    print(f"{split}: missing={len(missing)}")
    print(f"{split}: samples={truth.shape[1] - 1}")
    print(f"Wrote {output}")


make_split("test")
make_split("valid")